In [5]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"
start = NY_tz.localize(datetime.datetime(2026, 1, 7, 0, 0))
end = NY_tz.localize(datetime.datetime(2026, 1, 7, 23, 59))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
# df

MERGING SLICES...: 100%|██████████| 2/2 [00:00<00:00, 61.77it/s]


In [7]:
# from SDRUtils.products.usd.sofr_swaps import USD_SOFR_SwapProduct 
# USD_SOFR_SwapProduct().build_classification_dataframe(start=start, end=end, cache_path=cache_path)

from SDRUtils.products.usd.usd_swaptions import USD_Swaptions, straddle_pricer_from_row
sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path, ignore_cache=True)
# sdf.head(50)

Classifying Trades: 100%|██████████| 539/539 [00:00<00:00, 1638.62trade/s]


In [8]:
# sdf["product_type"].value_counts()
# sdf[(sdf["package_type"] == "STRADDLE") & ((sdf["forward_label"] == "3M")) & ((sdf["tenor_label"] == "10Y"))]

In [9]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP

mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-QL_BASIC")
pricer = mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))
pricer

QLIRSwapCurve(_ql_curve_id='USD-SOFR-1D', _ql_curve_handle=<QuantLib.QuantLib.YieldTermStructureHandle; proxy of <Swig Object of type 'Handle< YieldTermStructure > *' at 0x0000020EA0EAAD90> >, _ql_curve_index=<QuantLib.QuantLib.Sofr; proxy of <Swig Object of type 'ext::shared_ptr< Sofr > *' at 0x0000020E9C503120> >, _meta_data={'timestamp': datetime.datetime(2026, 1, 7, 0, 0)})

In [24]:
# sdf.loc[285], sdf.loc[286]
# .iloc[0].to_dict()
# df[df["Original Dissemination Identifier"] == 1653435994000001301]
straddle_pricer_from_row(sdf.loc[240], pricer)

(<QuantLib.QuantLib.Swaption; proxy of <Swig Object of type 'ext::shared_ptr< Swaption > *' at 0x0000020E9EA94450> >,
 33.7383388998905)

In [15]:
4.09 * np.sqrt(252)

np.float64(64.92673717352505)

In [19]:
# ["platform_identifier"].value_counts()
sdf[(sdf["package_type"] == "STRADDLE") & (sdf["package_indicator"] == False)]

,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,is_notional_capped,...,upi_underlier_name,unique_product_identifier,platform_identifier,cleared,package_indicator,package_transaction_price,option_premium_amount,package_confidence,package_reason,package_legs_count
240,NEWT-TRAD,1651551426000000601 / 1651573119000001801,2026-01-07 13:04:59+00:00,2026-01-07,2026-04-07,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-COMPOUND 1D CONSTANT 3Mx10Y PAYER EUR...,120000000.0,USD,False,...,NA/Swap Fxd Flt USD,QZXSN072GFF3 / QZXZSN00ZVCG,BILT,N,False,NaN,"1,344,002.24",1.0,platform=; time_delta_max=0.0s; premium_mode=S...,2
262,NEWT-NOVA,1653247770000002001 / 1653247773000002301,2026-01-07 14:43:20+00:00,2026-01-07,2027-10-06,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1Y CONSTANT 1Y9Mx1Y PAYE...,100000000.0,USD,False,...,NA/Swap OIS USD,QZZLNQ2D4JQT / QZJ92TTHTSF0,BILT,N,False,NaN,"470,000",1.0,platform=; time_delta_max=0.0s; premium_mode=S...,2
263,NEWT-TRAD,1652342172000000201 / 1652342173000000301,2026-01-07 14:43:20+00:00,2026-01-07,2027-10-06,SWAPTION_RECEIVER / SWAPTION_PAYER,USD-SOFR-OIS Compound 1Y CONSTANT 1Y9Mx1Y RECE...,200000000.0,USD,False,...,NA/Swap OIS USD,QZJ92TTHTSF0 / QZZLNQ2D4JQT,BILT,N,False,NaN,"940,000",1.0,platform=; time_delta_max=0.0s; premium_mode=S...,2
277,NEWT-TRAD,1652736044000000201 / 1652740760000000301,2026-01-07 15:46:32+00:00,2026-01-07,2026-02-09,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1Y CONSTANT 1Mx10Y PAYER...,27000000.0,USD,False,...,NA/Swap OIS USD,QZVLJBR2Z1VS / QZJ7QTJM4H97,BILT,N,False,NaN,"169,425",1.0,platform=; time_delta_max=0.0s; premium_mode=S...,2
286,NEWT-TRAD,1652844038000000101 / 1652930852000000601,2026-01-07 16:17:47+00:00,2026-01-07,2026-02-09,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 1Mx10Y PAYER...,20000000.0,USD,False,...,NA/Swap OIS USD,QZZGWPNBF5R3 / QZNLQ8T0N0SX,BILT,N,False,NaN,"128,000",1.0,platform=; time_delta_max=0.0s; premium_mode=S...,2


In [69]:
# df[df["Dissemination Identifier"] == 1570700287000000601].iloc[-1].to_dict()